# 🔬 Plastic Waste Detection — Full YOLOv8 Pipeline

**5-phase pipeline**: EDA → Data Prep → Training → Evaluation → Pseudo-Labeling → Export

## ⚙️ Setup Instructions
1. Upload your `ps_1_data/` folder as a **Kaggle Dataset** (with `img/`, `labels/`, `additional_images/` inside)
2. Add that dataset to this notebook via **Add Data** button
3. Set **Accelerator → GPU T4 x2** (or P100) in notebook settings
4. Update `DATASET_INPUT` path in the config cell below to match your dataset name
5. **Run All** cells sequentially

> **Important**: Kaggle input is read-only. We copy data to `/kaggle/working/` first.

## 📋 Configuration

Edit `DATASET_INPUT` to match the name of your uploaded Kaggle dataset.

In [ ]:
# ════════════════════════════════════════════════════════════
# CONFIGURATION — EDIT THIS SECTION
# ════════════════════════════════════════════════════════════

# Path to your uploaded dataset on Kaggle.
# After uploading ps_1_data as a dataset named e.g. 'ps-1-data',
# it appears at /kaggle/input/ps-1-data/ps_1_data/
# Update this path to match YOUR dataset name:
DATASET_INPUT = '/kaggle/input/ps-1-data/ps_1_data'

# Working directory (Kaggle writable area)
WORK_DIR = '/kaggle/working'

# Training hyperparameters
SEED = 42
IMG_SIZE = 640
BATCH_SIZE = 16        # Kaggle T4 can handle 16; reduce to 8 if OOM
PHASE2A_EPOCHS = 20    # Frozen backbone
PHASE2B_EPOCHS = 25    # Full fine-tune
ROUND2_EPOCHS = 30     # Pseudo-label retrain

# Class definitions
CLASS_NAMES = ['mix PP', 'mix HD', 'mix PET', 'mix rigid']
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

# ════════════════════════════════════════════════════════════

## 📦 Install Dependencies

In [ ]:
!pip install -q ultralytics albumentations tqdm pyyaml

## 🔧 Imports & Setup

In [ ]:
import os, sys, shutil, random, statistics, warnings
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import numpy as np
import torch
import yaml
import albumentations as A
from tqdm.auto import tqdm
from ultralytics import YOLO
from datetime import datetime

warnings.filterwarnings('ignore')
random.seed(SEED)
np.random.seed(SEED)

# Paths
DATA_DIR = Path(DATASET_INPUT)
IMG_DIR = DATA_DIR / 'img'
LABEL_DIR = DATA_DIR / 'labels'
ADDITIONAL_DIR = DATA_DIR / 'additional_images'
WORK = Path(WORK_DIR)
DATASET_DIR = WORK / 'dataset'
EDA_OUT = WORK / 'eda_outputs'
RUNS_DIR = WORK / 'runs'
PSEUDO_DIR = WORK / 'pseudo_labeled'
EDA_OUT.mkdir(parents=True, exist_ok=True)

# Device (Kaggle = CUDA GPU)
if torch.cuda.is_available():
    device = 'cuda'
    print(f'✓ Using CUDA GPU: {torch.cuda.get_device_name(0)}')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print('✓ Using MPS (Apple Metal)')
else:
    device = 'cpu'
    print('⚠ Using CPU — training will be slow')

print(f'Dataset: {DATA_DIR}')
print(f'Working: {WORK}')
assert DATA_DIR.exists(), f'Dataset not found at {DATA_DIR}! Update DATASET_INPUT.'

## 🛠️ Helper Functions

In [ ]:
def normalise_class(name):
    """Normalise class name to canonical form."""
    return {'mix pp':'mix PP','mix hd':'mix HD','mix pet':'mix PET','mix rigid':'mix rigid'}.get(name.strip().lower(), name.strip())

def extract_location(filename):
    """Extract location (SURAT/UMBERGAUM) from filename."""
    parts = filename.split('-')
    return parts[1].upper() if len(parts) >= 2 else 'UNKNOWN'

def parse_xml(xml_path):
    """Parse VOC XML → dict with filename, size, objects list."""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        size = root.find('size')
        w = int(size.findtext('width','640'))
        h = int(size.findtext('height','640'))
        objects = []
        for obj in root.findall('object'):
            name = normalise_class(obj.findtext('name',''))
            bb = obj.find('bndbox')
            objects.append({'name':name, 'xmin':int(float(bb.findtext('xmin','0'))),
                'ymin':int(float(bb.findtext('ymin','0'))), 'xmax':int(float(bb.findtext('xmax','0'))),
                'ymax':int(float(bb.findtext('ymax','0')))})
        return {'filename':root.findtext('filename',''), 'folder':root.findtext('folder',''),
                'width':w, 'height':h, 'objects':objects}
    except Exception as e:
        print(f'  ⚠ Failed to parse {xml_path.name}: {e}')
        return None

CLASS_COLOURS = {'mix PP':(0,255,0),'mix HD':(255,0,0),'mix PET':(0,0,255),'mix rigid':(0,255,255)}

---
# 🔍 PHASE 0 — Data Exploration (EDA)

In [ ]:
print('=' * 60)
print('PHASE 0 — DATA EXPLORATION')
print('=' * 60)

# File counts
image_exts = {'.jpeg','.jpg','.png'}
all_images = [f for f in IMG_DIR.iterdir() if f.suffix.lower() in image_exts and not f.name.startswith('.')]
all_xmls = sorted([f for f in LABEL_DIR.iterdir() if f.suffix.lower()=='.xml'])
xml_stems = {f.stem for f in all_xmls}
img_stems = {f.stem for f in all_images}
labeled = img_stems & xml_stems
unlabeled = img_stems - xml_stems

print(f'Images: {len(all_images)}, XMLs: {len(all_xmls)}, Labeled: {len(labeled)}, Unlabeled: {len(unlabeled)}')

# Additional images
additional = {}
if ADDITIONAL_DIR.exists():
    for sub in sorted(ADDITIONAL_DIR.iterdir()):
        if sub.is_dir() and not sub.name.startswith('.'):
            additional[sub.name] = len([f for f in sub.iterdir() if f.suffix.lower() in image_exts])
print(f'Additional unlabeled: {sum(additional.values())} — {additional}')

# Parse all XMLs
annotations = []
class_img = defaultdict(set); class_bbox = defaultdict(int)
loc_img = defaultdict(set); resolutions = set()
objs_per_img = []; bbox_stats = defaultdict(lambda:{'w':[],'h':[],'a':[]})

for xp in tqdm(all_xmls, desc='Parsing XMLs'):
    ann = parse_xml(xp)
    if not ann: continue
    annotations.append(ann)
    stem = xp.stem
    resolutions.add((ann['width'], ann['height']))
    loc = extract_location(ann['filename'])
    loc_img[loc].add(stem)
    objs_per_img.append(len(ann['objects']))
    classes_in = set()
    for o in ann['objects']:
        classes_in.add(o['name']); class_bbox[o['name']] += 1
        w,h = o['xmax']-o['xmin'], o['ymax']-o['ymin']
        bbox_stats[o['name']]['w'].append(w); bbox_stats[o['name']]['h'].append(h); bbox_stats[o['name']]['a'].append(w*h)
    for c in classes_in: class_img[c].add(stem)

# Report
print(f'\nClasses: {sorted(class_img.keys())}')
print(f'Resolutions: {resolutions}')
print(f'Locations: { {k:len(v) for k,v in loc_img.items()} }')
print(f'Objects/image: min={min(objs_per_img)}, max={max(objs_per_img)}, mean={statistics.mean(objs_per_img):.1f}')
for c in sorted(class_img):
    n = len(class_img[c])
    flag = ' ← CRITICAL' if n<10 else (' ← UNDERREP' if n<15 else '')
    print(f'  {c:<12}: {n:>3} images, {class_bbox[c]:>4} bboxes{flag}')

# Decision Report
total_labeled = len(labeled)
model_choice = 'yolov8n.pt' if total_labeled < 200 else ('yolov8s.pt' if total_labeled <= 500 else 'yolov8m.pt')
print(f'\n{"═"*50}')
print(f'DECISION: Model={model_choice}, Total labeled={total_labeled}')
print(f'Domain shift: YES — {list(loc_img.keys())}')
print(f'{"═"*50}')

# Draw 5 samples
samples = random.sample(annotations, min(5, len(annotations)))
for i,ann in enumerate(samples):
    ip = IMG_DIR / ann['filename']
    if not ip.exists(): continue
    img = cv2.imread(str(ip))
    if img is None: continue
    for o in ann['objects']:
        col = CLASS_COLOURS.get(o['name'],(255,255,255))
        cv2.rectangle(img,(o['xmin'],o['ymin']),(o['xmax'],o['ymax']),col,2)
        cv2.putText(img,o['name'],(o['xmin'],o['ymin']-4),cv2.FONT_HERSHEY_SIMPLEX,0.5,col,1)
    cv2.imwrite(str(EDA_OUT/f'sample_{i}.png'), img)
print(f'\n✓ Phase 0 complete — samples saved to {EDA_OUT}')

---
# 📦 PHASE 1 — Data Preparation

**1a** VOC→YOLO conversion | **1b** Stratified split | **1c** Augmentation

In [ ]:
print('=' * 60)
print('PHASE 1 — DATA PREPARATION')
print('=' * 60)

if DATASET_DIR.exists(): shutil.rmtree(DATASET_DIR)

# 1a. Convert VOC→YOLO and collect metadata
all_imgs_map = {f.stem:f for f in all_images}
data = {}
for xp in tqdm(all_xmls, desc='Converting VOC→YOLO'):
    stem = xp.stem
    if stem not in all_imgs_map: continue
    ann = parse_xml(xp)
    if not ann or not ann['objects']: continue
    w,h = ann['width'],ann['height']
    yolo_lines, voc_boxes = [], []
    for o in ann['objects']:
        if o['name'] not in CLASS_TO_ID: continue
        cid = CLASS_TO_ID[o['name']]
        x1,y1,x2,y2 = max(0,o['xmin']),max(0,o['ymin']),min(w,o['xmax']),min(h,o['ymax'])
        cx,cy = ((x1+x2)/2)/w, ((y1+y2)/2)/h
        bw,bh = (x2-x1)/w, (y2-y1)/h
        if bw>0 and bh>0:
            yolo_lines.append(f'{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
            voc_boxes.append((o['name'],int(x1),int(y1),int(x2),int(y2)))
    if yolo_lines:
        counts = Counter(int(l.split()[0]) for l in yolo_lines)
        data[stem] = {'yolo':yolo_lines,'voc':voc_boxes,'dom':counts.most_common(1)[0][0],
                      'loc':extract_location(ann['filename']),'img':all_imgs_map[stem],'w':w,'h':h}
print(f'✓ Converted {len(data)} images')

# 1b. Stratified split
umbergaum = [s for s,d in data.items() if d['loc']=='UMBERGAUM']
surat = [s for s,d in data.items() if d['loc']=='SURAT']
train_s, val_s, test_s = [], list(umbergaum), []
print(f'UMBERGAUM → val: {len(umbergaum)}')

groups = defaultdict(list)
for s in surat: groups[data[s]['dom']].append(s)
for cid in sorted(groups):
    stems = groups[cid]; random.shuffle(stems); n = len(stems)
    if n <= 15:
        if n>=3: test_s.append(stems[0]); val_s.append(stems[1]); train_s.extend(stems[2:])
        elif n==2: test_s.append(stems[0]); val_s.append(stems[1])
        else: train_s.append(stems[0])
    else:
        nt,nv = max(1,round(n*0.1)), max(1,round(n*0.1))
        test_s.extend(stems[:nt]); val_s.extend(stems[nt:nt+nv]); train_s.extend(stems[nt+nv:])
print(f'Split: train={len(train_s)}, val={len(val_s)}, test={len(test_s)}')

# Write splits
def write_split(stems, split):
    idir = DATASET_DIR/split/'images'; ldir = DATASET_DIR/split/'labels'
    idir.mkdir(parents=True,exist_ok=True); ldir.mkdir(parents=True,exist_ok=True)
    for s in stems:
        d = data[s]
        shutil.copy2(str(d['img']), str(idir/d['img'].name))
        (ldir/(d['img'].stem+'.txt')).write_text('\n'.join(d['yolo'])+'\n')

for stems,split in [(train_s,'train'),(val_s,'val'),(test_s,'test')]:
    write_split(stems, split)

# 1c. Augmentation
class_train_counts = Counter(data[s]['dom'] for s in train_s)

def get_aug(severity='normal'):
    t = [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.3), A.RandomRotate90(p=0.3),
         A.Rotate(limit=30, border_mode=cv2.BORDER_CONSTANT, p=0.4),
         A.RandomBrightnessContrast(brightness_limit=0.4,contrast_limit=0.4,p=0.5),
         A.HueSaturationValue(hue_shift_limit=10,sat_shift_limit=30,val_shift_limit=30,p=0.4),
         A.GaussianBlur(blur_limit=(3,7),p=0.3), A.GaussNoise(p=0.3)]
    if severity=='heavy':
        t.extend([A.ElasticTransform(alpha=120,sigma=6,p=0.3),A.Perspective(scale=(0.05,0.1),p=0.3)])
    return A.Compose(t, bbox_params=A.BboxParams(format='pascal_voc',min_visibility=0.3,label_fields=['class_labels']))

normal_aug, heavy_aug = get_aug('normal'), get_aug('heavy')
idir = DATASET_DIR/'train'/'images'; ldir = DATASET_DIR/'train'/'labels'
total_aug = 0; verify_imgs = []

for stem in tqdm(train_s, desc='Augmenting'):
    d = data[stem]; cnt = class_train_counts[d['dom']]
    mult = 10 if cnt<10 else (8 if cnt<20 else 5)
    pipe = heavy_aug if cnt<10 else normal_aug
    img = cv2.imread(str(d['img'])); 
    if img is None: continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    bboxes = [[max(0,min(x1,d['w']-1)),max(0,min(y1,d['h']-1)),max(1,min(x2,d['w'])),max(1,min(y2,d['h']))] for nm,x1,y1,x2,y2 in d['voc']]
    labels = [nm for nm,_,_,_,_ in d['voc']]
    for ai in range(mult):
        try:
            r = pipe(image=img_rgb, bboxes=bboxes, class_labels=labels)
            if not r['bboxes']: continue
            h2,w2 = r['image'].shape[:2]
            yl = []; 
            for bb,lb in zip(r['bboxes'],r['class_labels']):
                x1,y1,x2,y2 = bb; cid = CLASS_TO_ID[lb]
                cx,cy,bw,bh = ((x1+x2)/2)/w2,((y1+y2)/2)/h2,(x2-x1)/w2,(y2-y1)/h2
                if bw>0 and bh>0: yl.append(f'{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
            if not yl: continue
            name = f'{stem}_aug{ai}'
            cv2.imwrite(str(idir/f'{name}.jpeg'), cv2.cvtColor(r['image'],cv2.COLOR_RGB2BGR))
            (ldir/f'{name}.txt').write_text('\n'.join(yl)+'\n')
            total_aug += 1
            if len(verify_imgs)<8: verify_imgs.append((cv2.cvtColor(r['image'],cv2.COLOR_RGB2BGR),r['bboxes'],r['class_labels']))
        except: continue

print(f'✓ {total_aug} augmented images')

# Verification mosaic
if verify_imgs:
    mosaic = np.zeros((2*320,4*320,3),dtype=np.uint8)
    for idx,(im,bbs,lbs) in enumerate(verify_imgs[:8]):
        im2 = cv2.resize(im,(320,320)); ho,wo = im.shape[:2]
        for bb,lb in zip(bbs,lbs):
            x1,y1,x2,y2 = bb
            cv2.rectangle(im2,(int(x1*320/wo),int(y1*320/ho)),(int(x2*320/wo),int(y2*320/ho)),CLASS_COLOURS.get(lb,(255,255,255)),2)
        r,c = idx//4, idx%4
        mosaic[r*320:(r+1)*320, c*320:(c+1)*320] = im2
    cv2.imwrite(str(EDA_OUT/'augmentation_verification.png'), mosaic)

# dataset.yaml
yaml_path = WORK / 'dataset.yaml'
yaml_path.write_text(f"""path: {DATASET_DIR.resolve()}
train: train/images
val: val/images
test: test/images
nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
""")

ti = len(list((DATASET_DIR/'train'/'images').glob('*')))
vi = len(list((DATASET_DIR/'val'/'images').glob('*')))
tsi = len(list((DATASET_DIR/'test'/'images').glob('*')))
print(f'Final dataset: train={ti}, val={vi}, test={tsi}')
print('\n✓ Phase 1 complete')

---
# 🏋️ PHASE 2 — Training

**2a**: Frozen backbone (20 epochs) → **2b**: Full fine-tune (25 epochs)

In [ ]:
print('=' * 60)
print('PHASE 2a — FROZEN BACKBONE')
print('=' * 60)

yaml_path = WORK / 'dataset.yaml'

model = YOLO(model_choice)
model.train(
    data=str(yaml_path), epochs=PHASE2A_EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE,
    lr0=1e-3, optimizer='AdamW', freeze=10,
    mosaic=1.0, mixup=0.1, copy_paste=0.1, dropout=0.1, patience=10,
    device=device, seed=SEED,
    project=str(RUNS_DIR), name='phase1_frozen', exist_ok=True, verbose=True
)

best_frozen = RUNS_DIR / 'phase1_frozen' / 'weights' / 'best.pt'
print(f'\n✓ Phase 2a complete — {best_frozen}')

In [ ]:
print('=' * 60)
print('PHASE 2b — FULL FINE-TUNE')
print('=' * 60)

model = YOLO(str(best_frozen))
model.train(
    data=str(yaml_path), epochs=PHASE2B_EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE,
    lr0=1e-4, optimizer='AdamW', freeze=0,
    mosaic=1.0, mixup=0.15, copy_paste=0.15, dropout=0.1, patience=15,
    device=device, seed=SEED,
    project=str(RUNS_DIR), name='phase2_finetune', exist_ok=True, verbose=True
)

best_ft = RUNS_DIR / 'phase2_finetune' / 'weights' / 'best.pt'
print(f'\n✓ Phase 2b complete — {best_ft}')

---
# 📊 PHASE 3 — Evaluation

In [ ]:
print('=' * 60)
print('PHASE 3 — EVALUATION')
print('=' * 60)

best_model = best_ft if best_ft.exists() else best_frozen
model = YOLO(str(best_model))

for split, label in [('test','TEST (SURAT)'), ('val','VAL (UMBERGAUM)')]:
    r = model.val(data=str(yaml_path), split=split, verbose=True, plots=True,
                  project=str(RUNS_DIR), name=f'eval_{split}', exist_ok=True)
    print(f'\n{label}: mAP@0.5={r.box.map50:.4f}, mAP@0.5:0.95={r.box.map:.4f}')
    if hasattr(r.box,'ap_class_index') and r.box.ap_class_index is not None:
        for i,ci in enumerate(r.box.ap_class_index):
            cn = CLASS_NAMES[int(ci)] if int(ci)<len(CLASS_NAMES) else f'cls_{ci}'
            print(f'  {cn:<12}: P={r.box.p[i]:.3f} R={r.box.r[i]:.3f} AP50={r.box.ap50[i]:.3f}')

# Copy plots
for split in ['test','val']:
    edir = RUNS_DIR/f'eval_{split}'
    if edir.exists():
        for f in edir.glob('*.png'): shutil.copy2(str(f), str(EDA_OUT/f'eval_{split}_{f.name}'))

print('\n✓ Phase 3 complete')

---
# 🏷️ PHASE 4 — Pseudo-Labeling (Semi-supervised)

In [ ]:
print('=' * 60)
print('PHASE 4 — PSEUDO-LABELING')
print('=' * 60)

CONF_THRESH = {'mix PP':0.80,'mix HD':0.70,'mix PET':0.80,'mix rigid':0.80}
best_model = best_ft if best_ft.exists() else best_frozen
model = YOLO(str(best_model))

pimg = PSEUDO_DIR/'images'; plbl = PSEUDO_DIR/'labels'
pimg.mkdir(parents=True,exist_ok=True); plbl.mkdir(parents=True,exist_ok=True)

add_imgs = []
for sub in sorted(ADDITIONAL_DIR.iterdir()):
    if sub.is_dir() and not sub.name.startswith('.'):
        add_imgs.extend(sorted([f for f in sub.iterdir() if f.suffix.lower() in image_exts]))

accepted, rejected = 0, 0
cls_acc, cls_tot = Counter(), Counter()

for ip in tqdm(add_imgs, desc='Pseudo-labeling'):
    # Original inference
    r1 = model.predict(source=str(ip), device=device, conf=0.5, verbose=False)
    dets = [(int(b.cls[0]),CLASS_NAMES[int(b.cls[0])],float(b.conf[0]),*b.xyxy[0].tolist()) for r in r1 for b in (r.boxes if r.boxes is not None else [])]
    if not dets: rejected+=1; continue
    # Flip consistency check
    try:
        im = cv2.imread(str(ip)); flipped = cv2.flip(im,1)
        tp = Path('/tmp/pf.jpeg'); cv2.imwrite(str(tp),flipped)
        r2 = model.predict(source=str(tp),device=device,conf=0.5,verbose=False)
        tp.unlink(missing_ok=True)
        orig_cls = set(d[1] for d in dets)
        flip_cls = set(CLASS_NAMES[int(b.cls[0])] for r in r2 for b in (r.boxes if r.boxes is not None else []))
        consistent = orig_cls & flip_cls
        dets = [d for d in dets if d[1] in consistent]
    except: pass
    if not dets: rejected+=1; continue
    # Confidence filter
    good = []
    for d in dets:
        cls_tot[d[1]] += 1
        if d[2] >= CONF_THRESH.get(d[1],0.80): good.append(d); cls_acc[d[1]] += 1
    if not good: rejected+=1; continue
    # Save
    im = cv2.imread(str(ip))
    if im is None: continue
    h2,w2 = im.shape[:2]
    shutil.copy2(str(ip), str(pimg/ip.name))
    yl = []
    for cid,cn,cf,x1,y1,x2,y2 in good:
        cx,cy,bw,bh = ((x1+x2)/2)/w2,((y1+y2)/2)/h2,(x2-x1)/w2,(y2-y1)/h2
        if bw>0 and bh>0: yl.append(f'{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
    if yl: (plbl/(ip.stem+'.txt')).write_text('\n'.join(yl)+'\n'); accepted+=1

print(f'\nAccepted: {accepted}, Rejected: {rejected}')
low_acc = []
for cn in CLASS_NAMES:
    t,a = cls_tot[cn],cls_acc[cn]
    rate = (a/t*100) if t>0 else 0
    print(f'  {cn:<12}: {a}/{t} ({rate:.1f}%)')
    if rate < 20 and t > 0: low_acc.append(cn)
if low_acc: print(f'⚠ Low acceptance (<20%): {low_acc} — excluded from Round 2')

### Round 2 Training — Original + Pseudo-labels

In [ ]:
# Combine original train + pseudo-labels
r2dir = WORK / 'dataset_round2'
if r2dir.exists(): shutil.rmtree(r2dir)
shutil.copytree(str(DATASET_DIR), str(r2dir))

r2img = r2dir/'train'/'images'; r2lbl = r2dir/'train'/'labels'
p_added = 0
for lf in plbl.glob('*.txt'):
    skip = False
    if low_acc:
        for line in lf.read_text().strip().split('\n'):
            cid = int(line.split()[0])
            if cid<len(CLASS_NAMES) and CLASS_NAMES[cid] in low_acc: skip=True; break
    if skip: continue
    for ext in ['.jpeg','.jpg','.png']:
        si = pimg/(lf.stem+ext)
        if si.exists(): shutil.copy2(str(si),str(r2img/si.name)); shutil.copy2(str(lf),str(r2lbl/lf.name)); p_added+=1; break

print(f'Round 2: original + {p_added} pseudo-labels')

r2yaml = WORK/'dataset_round2.yaml'
r2yaml.write_text(f"""path: {r2dir.resolve()}
train: train/images
val: val/images
test: test/images
nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
""")

model = YOLO(str(best_ft if best_ft.exists() else best_frozen))
model.train(
    data=str(r2yaml), epochs=ROUND2_EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE,
    lr0=5e-5, optimizer='AdamW', freeze=0,
    mosaic=1.0, mixup=0.15, copy_paste=0.15, dropout=0.1, patience=15,
    device=device, seed=SEED,
    project=str(RUNS_DIR), name='round2_pseudo', exist_ok=True, verbose=True
)

r2best = RUNS_DIR/'round2_pseudo'/'weights'/'best.pt'
print(f'\n✓ Phase 4 complete — Round 2 best: {r2best}')

---
# 📤 PHASE 5 — Export & Final Report

In [ ]:
print('=' * 60)
print('PHASE 5 — EXPORT')
print('=' * 60)

# Find overall best
for p in [RUNS_DIR/'round2_pseudo'/'weights'/'best.pt', best_ft, best_frozen]:
    if p.exists(): final_best = p; break

model = YOLO(str(final_best))
onnx_path = model.export(format='onnx', optimize=True, simplify=True)
print(f'✓ ONNX exported: {onnx_path}')

# Final metrics
r = model.val(data=str(yaml_path), split='val', verbose=False, project=str(RUNS_DIR), name='final_eval', exist_ok=True)
metrics = {'mAP50':float(r.box.map50), 'mAP50_95':float(r.box.map)}

# model_info.yaml
info = {
    'architecture':'YOLOv8n','classes':CLASS_NAMES,'input_size':IMG_SIZE,
    'final_metrics':metrics, 'best_weights':str(final_best), 'onnx':str(onnx_path),
    'training':{'phase2a':'20ep frozen lr=1e-3','phase2b':'25ep full lr=1e-4','round2':'30ep pseudo lr=5e-5'}
}
(WORK/'model_info.yaml').write_text(yaml.dump(info, default_flow_style=False))
print(f'✓ model_info.yaml saved')
print(f'\nFinal mAP@0.5: {metrics["mAP50"]:.4f}')
print(f'Final mAP@0.5:0.95: {metrics["mAP50_95"]:.4f}')
print('\n✓ Phase 5 complete')
print('✓ FULL PIPELINE COMPLETE — all outputs in /kaggle/working/')

---
# 📥 Download Outputs

After running, download key files from `/kaggle/working/`:
- `runs/round2_pseudo/weights/best.pt` — Best model weights
- `model_info.yaml` — Training summary
- `eda_outputs/` — Visualisations
- The ONNX export file for deployment